In [ ]:
import pandas as pd
import glob
import os
from tqdm import tqdm
import numpy as np
import re
from datetime import datetime

FILE_INPUT_1 = "PERCORSO_FILE_ANAGRAFICA.xlsx"
CARTELLA_INPUT_2 = "PERCORSO_CARTELLE_LETTURE"
FILE_INPUT_DB = "DB_DIZIONARIO.parquet"
DIR_SORGENTE_DATI = "PERCORSO_SORGENTE_SETTIMANALE"

def formatta_colonne(df):
    df.columns = df.columns.str.upper().str.strip()
    for col in df.select_dtypes(include=[object]).columns:
        df[col] = df[col].map(lambda x: x.upper().strip() if isinstance(x, str) else x)
    return df

def normalizza_codici(x):
    if pd.isna(x) or x == '': return x
    try:
        return str(int(float(x)))
    except (ValueError, TypeError):
        return str(x).strip()

def pulizia_stringhe_codice(df, col):
    df[col] = df[col].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    return df

def calcola_stato_metrica(valore):
    if pd.isna(valore): return 'N/D'
    if valore == 0: return 'LIVELLO_0'
    if 0 < valore < 49: return "FASCIA_D"
    if 49 <= valore < 80: return "FASCIA_C"
    if 80 <= valore <= 100: return "FASCIA_B"
    if valore > 100: return "FASCIA_A"
    return 'EXTRARANGE'

print(f"Acquisizione flussi di dati...")
estensioni_file = glob.glob(os.path.join(CARTELLA_INPUT_2, "*.xls*"))
lista_acquisizione = []

for f in estensioni_file:
    try:
        df_temp = pd.read_excel(f)
        df_temp['ID_ORIGINE'] = os.path.basename(f)
        lista_acquisizione.append(df_temp)
    except Exception as e:
        print(f"Errore caricamento: {e}")

if lista_acquisizione:
    DF_1 = pd.concat(lista_acquisizione, ignore_index=True)
else:
    DF_1 = pd.DataFrame()

DF_1 = formatta_colonne(DF_1)

CODICE_FILTRO = 'PARAM_01'
DF_1_A = DF_1[DF_1['CODICE_OPERATIVO'] == CODICE_FILTRO].copy()
DF_1_B = DF_1[DF_1['CODICE_OPERATIVO'] != CODICE_FILTRO].copy()

for df_proc in [DF_1_A, DF_1_B]:
    df_proc['DATA_RIF'] = pd.to_datetime(df_proc['DATA_RIF'], dayfirst=True)
    df_proc['PERIODO_SETT'] = df_proc['DATA_RIF'].dt.isocalendar().week.astype(int)

SETTIMANE_RIF = DF_1_A['PERIODO_SETT'].unique().tolist()

DF_2 = pd.read_parquet(FILE_INPUT_DB)
DF_2 = formatta_colonne(DF_2)
DF_2 = DF_2[DF_2['CATEGORIA_RIF'] == 'CAT_A'].copy()
DF_2 = pulizia_stringhe_codice(DF_2, 'ID_CHIAVE')

file_settimanali = glob.glob(os.path.join(DIR_SORGENTE_DATI, "*.xlsx"))
lista_df_sett = []

for f in tqdm(file_settimanali, desc="Processing flussi"):
    match = re.match(r"^(\d+)_", os.path.basename(f))
    if match:
        val_sett = int(match.group(1))
        if val_sett in SETTIMANE_RIF:
            df_temp = pd.read_excel(f, header=2) 
            df_temp['PERIODO_SETT'] = val_sett
            lista_df_sett.append(df_temp)

DF_3 = pd.concat(lista_df_sett, ignore_index=True) if lista_df_sett else pd.DataFrame()
DF_3 = formatta_colonne(DF_3)

MAPPING_COL = {
    'CODICE_PRIMARIO': 'ID_CHIAVE',
    'VOCE_1': 'DESCRIZIONE_VOCE',
    'PARAM_A': 'FLAG_A',
    'PARAM_B': 'METRICA_FREQUENZA',
    'PARAM_C': 'FLAG_B'
}
DF_3 = DF_3.rename(columns=MAPPING_COL)

DF_3.loc[DF_3['ID_CHIAVE'].isin(['#N/D', '']), 'ID_CHIAVE'] = np.nan
maschera_null = DF_3['ID_CHIAVE'].isna()
if maschera_null.any():
    DF_3['ID_CHIAVE'] = np.where(maschera_null, 'GENERIC_' + maschera_null.cumsum().astype(str), DF_3['ID_CHIAVE'])

DF_3 = DF_3[DF_3['FLAG_A'].isna() & DF_3['METRICA_FREQUENZA'].notna() & DF_3['FLAG_B'].isna()].copy()
DF_3 = pulizia_stringhe_codice(DF_3, 'ID_CHIAVE')

DATA_REPORT = []

for s in sorted(DF_3['PERIODO_SETT'].unique()):
    DF_3_S = DF_3[DF_3['PERIODO_SETT'] == s].copy()
    DF_1_A_S = DF_1_A[DF_1_A['PERIODO_SETT'] == s].copy()
    DF_1_B_S = DF_1_B[DF_1_B['PERIODO_SETT'] == s].copy()
    
    ids_previsti = DF_3_S['ID_CHIAVE'].unique().tolist()
    
    DF_M1 = pd.merge(DF_3_S, DF_2, on='ID_CHIAVE', how='inner')
    presenti_db = DF_M1['ID_CHIAVE'].unique().tolist()

    DF_M2 = pd.merge(DF_M1, DF_1_A_S, left_on='TAG_ID', right_on='TAG_RILEVATO', how='inner')
    rilevati_target = DF_M2['ID_CHIAVE'].unique().tolist()

    DF_M3 = pd.merge(DF_M1, DF_1_B_S, left_on='TAG_ID', right_on='TAG_RILEVATO', how='inner')
    rilevati_extra = [c for c in DF_M3['ID_CHIAVE'].unique() if c not in rilevati_target]

    DATA_REPORT.append({
        'PERIODO': s,
        'TOT_ID_PREVISTI': len(ids_previsti),
        'MATCH_DB': len(presenti_db),
        'RILEVATI_IN': len(rilevati_target),
        'RILEVATI_OUT': len(rilevati_extra),
        'NON_IN_DB': len([c for c in ids_previsti if c not in presenti_db])
    })

DF_REPORT_FINAL = pd.DataFrame(DATA_REPORT)
DF_REPORT_FINAL['PERC_COPERTURA'] = ((DF_REPORT_FINAL['RILEVATI_IN'] / DF_REPORT_FINAL['TOT_ID_PREVISTI']) * 100).round(2)
DF_REPORT_FINAL['INDICATORE_STATO'] = DF_REPORT_FINAL['PERC_COPERTURA'].apply(calcola_stato_metrica)

DF_ANALISI_DETTAGLIO = []

for s in sorted(DF_3['PERIODO_SETT'].unique()):
    DF_3_S = DF_3[DF_3['PERIODO_SETT'] == s].copy()
    DF_1_A_S = DF_1_A[DF_1_A['PERIODO_SETT'] == s].copy()
    DF_1_B_S = DF_1_B[DF_1_B['PERIODO_SETT'] == s].copy()
    
    DF_BASE_S = pd.merge(DF_3_S, DF_2, on='ID_CHIAVE', how='inner')
    
    C1 = pd.merge(DF_BASE_S, DF_1_A_S, left_on='TAG_ID', right_on='TAG_RILEVATO', how='inner').groupby('ID_CHIAVE').size().reset_index(name='COUNT_IN')
    C2 = pd.merge(DF_BASE_S, DF_1_B_S, left_on='TAG_ID', right_on='TAG_RILEVATO', how='inner').groupby('ID_CHIAVE').size().reset_index(name='COUNT_OUT')

    DF_STEP = pd.merge(DF_BASE_S, C1, on='ID_CHIAVE', how='left').merge(C2, on='ID_CHIAVE', how='left')
    DF_STEP[['COUNT_IN', 'COUNT_OUT']] = DF_STEP[['COUNT_IN', 'COUNT_OUT']].fillna(0).astype(int)
    
    DF_ANALISI_DETTAGLIO.append(DF_STEP)

DF_MASTER_DETTAGLIO = pd.concat(DF_ANALISI_DETTAGLIO, ignore_index=True)

DF_ANOMALIE = DF_MASTER_DETTAGLIO[(DF_MASTER_DETTAGLIO['COUNT_IN'] == 0) & (DF_MASTER_DETTAGLIO['COUNT_OUT'] == 0)].copy()
CONTEGGIO_ASSENZE = DF_ANOMALIE.groupby('ID_CHIAVE')['PERIODO_SETT'].nunique().reset_index(name='SETTIMANE_ASSENZA')
DF_ANOMALIE = DF_ANOMALIE.merge(CONTEGGIO_ASSENZE, on='ID_CHIAVE', how='left')

PATH_OUTPUT = "REPORT_SINTESI_OPERATIVA.xlsx"

CONFIG_OUTPUT = {
    "SINTESI_PERFORMANCE": DF_REPORT_FINAL,
    "DETTAGLIO_ANOMALIE": DF_ANOMALIE,
    "MASTER_DATA_RILEVATO": DF_MASTER_DETTAGLIO
}

with pd.ExcelWriter(PATH_OUTPUT, engine='xlsxwriter') as writer:
    for nome_foglio, df_foglio in CONFIG_OUTPUT.items():
        df_foglio.to_excel(writer, sheet_name=nome_foglio, index=False)
        
print(f"Processo completato. File generato: {PATH_OUTPUT}")

In [ ]:
Questo script è un esempio di automazione ETL (Extract, Transform, Load) per l'analisi dei dati aziendali. In sintesi, il codice integra diverse sorgenti (Excel e Parquet), pulisce e normalizza i dati per renderli coerenti, e incrocia le informazioni per monitorare la "copertura" delle rilevazioni settimanali rispetto a un database di riferimento, identificando anomalie e assenze.

L'esecuzione dello script dimostra una solida padronanza delle seguenti competenze tecniche:

1. Manipolazione Dati con Pandas
Gestione di Dataset Multipli: Capacità di unire (merge) e concatenare (concat) dataframe provenienti da fonti diverse usando logiche di tipo SQL (inner e left join).

Data Cleaning Avanzato: Uso di espressioni regolari (re), gestione dei valori nulli (NaN) e normalizzazione di stringhe e tipi di dato.

Aggregazione e Analisi: Utilizzo di groupby, size() e nunique() per estrarre metriche statistiche e contatori dai dati grezzi.

2. Gestione del File System e Automazione
I/O Multi-formato: Lettura e scrittura di file .xlsx (anche con motori specifici come xlsxwriter) e formati performanti come .parquet.

Iterazione Massiva: Utilizzo di glob e os per scansionare intere cartelle e processare file in batch, ottimizzando il tempo con barre di progressione (tqdm).

3. Logica di Business e Problem Solving
Analisi Temporale: Gestione delle date tramite l'oggetto datetime e conversione in settimane ISO per il monitoraggio periodico.

Segmentazione (Binning): Creazione di funzioni custom per classificare le performance in fasce di qualità (Fascia A, B, C, D).

Identificazione Anomalie: Capacità di tradurre un problema di business (es. "cosa manca nelle rilevazioni?") in una maschera logica di filtraggio dati.

4. Strutturazione del Codice
Modularità: Suddivisione del lavoro in funzioni riutilizzabili (formatta_colonne, calcola_stato_metrica).

Configurabilità: Utilizzo di costanti a inizio script per definire percorsi e parametri, facilitando la manutenzione del software.